In [47]:
import torch
import torchcodec
from pathlib import Path
from datasets import Dataset, DatasetDict, Audio

# print(torch.__version__)
# print(torchcodec.__version__)

01. Environment Setup
02. Explore AISHELL Dataset
03. Match Audio and Transcriptions
04. Build Dataset with Generators
05. Create Hanzi and Pinyin Datasets
06. Data Cleaning and Validation
07. Build CTC Vocabularies
08. Initialize Wav2Vec2 Processors
09. Prepare Audio and Labels
10. Fine-tune Hanzi ASR
11. Fine-tune Pinyin ASR
12. Evaluation and Comparison

# Environmental Setup and Get Files

In [48]:
# get the project root and the data root
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "aishell3"

print(PROJECT_ROOT)
print(DATA_ROOT)

# root: 
# /Users/shuangping/Desktop/ASR_Project
# /Users/shuangping/Desktop/ASR_Project/aishell3

/Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR
/Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/aishell3


# Build HF dataset, including hanzi set and pinyin set

In [49]:
import re

HANZI_RE = re.compile(r"^[\u4e00-\u9fff]+$") # utf-8
PINYIN_RE = re.compile(r"^[a-zA-ZüÜ]+[1-5]?$") # ascii letters with optional tone numbers

def is_hanzi(token: str) -> bool:
    return bool(HANZI_RE.fullmatch(token))

def is_pinyin(token: str) -> bool:
    return bool(PINYIN_RE.fullmatch(token))

def parse_aishell3_line(line: str):
    parts = line.strip().split()
    if not parts:
        return None

    wav_name = parts[0]
    tokens = parts[1:]

    hanzi_chars = []
    pinyin_chars = []

    for tok in tokens:
        if is_hanzi(tok):
            hanzi_chars.append(tok)
        elif is_pinyin(tok):
            pinyin_chars.append(tok)
        else:
            raise ValueError(f"Unsupported token: {tok!r} in line: {line.rstrip()}")

    hanzi = "".join(hanzi_chars)
    pinyin = " ".join(pinyin_chars)

    return wav_name, hanzi, pinyin

In [50]:
# define the generator
def aishell3_generator(content_file, wav_root):
    content_file = Path(content_file)
    wav_root = Path(wav_root)

    with content_file.open("r", encoding="utf-8") as f:
        for _, line in enumerate(f, start=1):
            if not line.strip():
                continue

            parsed = parse_aishell3_line(line)
            if parsed is None:
                continue

            wav_name, hanzi, pinyin = parsed

            speaker_id = Path(wav_name).stem[:7]
            audio_path = wav_root / speaker_id / wav_name

            if not audio_path.exists():
                raise FileNotFoundError(f"Audio Not Found: {audio_path}")

            yield {
                "id": Path(wav_name).stem,
                "speaker_id": speaker_id,
                "audio": str(audio_path.resolve()),
                "hanzi": hanzi,
                "pinyin": pinyin,
            }

In [51]:
train_dataset = Dataset.from_generator(
    aishell3_generator,
    gen_kwargs={
        "content_file": str(DATA_ROOT / "train" / "content.txt"), # get the train data from train set
        "wav_root": str(DATA_ROOT / "train" / "wav"),
    },
)

test_dataset = Dataset.from_generator(
    aishell3_generator,
    gen_kwargs={
        "content_file": str(DATA_ROOT / "test" / "content.txt"), #get the test data from the test set
        "wav_root": str(DATA_ROOT / "test" / "wav"),
    },
)

aishell_base = DatasetDict({
    "train": train_dataset,
    "test": test_dataset,
})

aishell_base

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'hanzi', 'pinyin'],
        num_rows: 63262
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'hanzi', 'pinyin'],
        num_rows: 24773
    })
})

In [52]:
# Cast the "audio" column to the Audio type, set sample rate to 16kHz and number of channels to 1 (mono)
aishell_base = aishell_base.cast_column("audio", Audio(sampling_rate=16_000, num_channels=1),)

In [53]:
sample = aishell_base["train"][0]

print("ID:", sample["id"])
print("Speaker:", sample["speaker_id"])
print("Hanzi:", sample["hanzi"])
print("Pinyin:", sample["pinyin"])

ID: SSB00050001
Speaker: SSB0005
Hanzi: 广州女大学生登山失联四天警方找到疑似女尸
Pinyin: guang3 zhou1 nv3 da4 xue2 sheng1 deng1 shan1 shi1 lian2 si4 tian1 jing3 fang1 zhao3 dao4 yi2 si4 nv3 shi1


In [54]:
# check the audio data, including sample rate and audio length
audio = sample["audio"]

print(audio)
print("sample_rate:", audio["sampling_rate"])
print("audio_length:", len(audio["array"]))

sample_rate: 16000
audio_length: 151109


# Get Speaker Meta Data

In [55]:
import csv

metadata_path = DATA_ROOT / "spk-info.txt"

speaker_metadata = {}

with metadata_path.open("r", encoding="utf-8") as f:

    reader = csv.reader(f, delimiter="\t")

    for line_number, row in enumerate(reader, start=1):

        if not row or not any(value.strip() for value in row):

            continue

        if row[0].strip().startswith("#"):

            continue

        row = [value.strip() for value in row]

        if len(row) != 4:

            raise ValueError(

                f"Invalid metadata row {line_number}: "

                f"expected 4 columns, got {len(row)}\n"

                f"Row: {row}"

            )

        speaker_id, age_group, gender, accent = row

        speaker_id = speaker_id.upper()

        if age_group not in {"A", "B", "C", "D"}:

            raise ValueError(

                f"Unknown age group '{age_group}' "

                f"for speaker {speaker_id}"

            )

        if speaker_id in speaker_metadata:

            raise ValueError(

                f"Duplicate metadata for speaker: {speaker_id}"

            )

        speaker_metadata[speaker_id] = {

            "age_group": age_group,

            "gender": gender.lower(),

            "accent": accent.lower()

        }

print("Speakers in metadata:", len(speaker_metadata))

Speakers in metadata: 218


In [56]:
dataset_speakers = set()

for split_dataset in aishell_base.values():

    dataset_speakers.update(split_dataset.unique("speaker_id"))

metadata_speakers = set(speaker_metadata)

missing_metadata = dataset_speakers - metadata_speakers

unused_metadata = metadata_speakers - dataset_speakers

print("Speakers in dataset:", len(dataset_speakers))

print("Speakers without metadata:", sorted(missing_metadata))

print("Unused metadata speakers:", len(unused_metadata))

if missing_metadata:

    raise ValueError(

        f"{len(missing_metadata)} speakers have no metadata: "

        f"{sorted(missing_metadata)}"

    )

Speakers in dataset: 218
Speakers without metadata: []
Unused metadata speakers: 0


In [57]:
def add_speaker_metadata(speaker_id):

    speaker_id = speaker_id.upper()

    return speaker_metadata[speaker_id]

aishell_base = aishell_base.map(

    add_speaker_metadata,

    input_columns="speaker_id",

    desc="Adding speaker metadata",

)

print(aishell_base["train"][0])

{'id': 'SSB00050001', 'speaker_id': 'SSB0005', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x1608ed5a0>, 'hanzi': '广州女大学生登山失联四天警方找到疑似女尸', 'pinyin': 'guang3 zhou1 nv3 da4 xue2 sheng1 deng1 shan1 shi1 lian2 si4 tian1 jing3 fang1 zhao3 dao4 yi2 si4 nv3 shi1', 'age_group': 'B', 'gender': 'female', 'accent': 'north'}


In [58]:
# Split 10% of the original training set as validation set

train_validation_split = aishell_base["train"].train_test_split(
    test_size=0.1,
    seed=42,
    shuffle=True,
)

aishell_base = DatasetDict({
    "train": train_validation_split["train"],
    "validation": train_validation_split["test"],
    "test": aishell_base["test"],
})

print(aishell_base)
print("Train:", len(aishell_base["train"]))
print("Validation:", len(aishell_base["validation"]))
print("Test:", len(aishell_base["test"]))

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})
Train: 56935
Validation: 6327
Test: 24773


# Get the Hanzi and Pinyin dataset

In [59]:
# Remove the "pinyin" column and rename "hanzi" to "text"
hanzi_dataset = aishell_base.remove_columns(["pinyin"])
hanzi_dataset = hanzi_dataset.rename_column("hanzi", "text")

print(hanzi_dataset)
print(hanzi_dataset["train"][0])
print(hanzi_dataset["validation"][0])
print(hanzi_dataset["test"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})
{'id': 'SSB03820023', 'speaker_id': 'SSB0382', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x1608eda70>, 'text': '但这一点已经足够表明卷福的超高人气', 'age_group': 'B', 'gender': 'female', 'accent': 'north'}
{'id': 'SSB03410187', 'speaker_id': 'SSB0341', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x11cc30cd0>, 'text': '中联航将被削减百分之十的飞行小时数', 'age_group': 'C', 'gender': 'female', 'accent': 'north'}
{'id': 'SSB06930002', 'speaker_id': 'SSB0693', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x11cc30cd0>, 'text': '武

In [60]:
# Remove the "hanzi" column and rename "pinyin" to "text"
pinyin_dataset = aishell_base.remove_columns(["hanzi"])
pinyin_dataset = pinyin_dataset.rename_column("pinyin", "text")

print(pinyin_dataset)
print(pinyin_dataset["train"][0])
print(pinyin_dataset["validation"][0])
print(pinyin_dataset["test"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})
{'id': 'SSB03820023', 'speaker_id': 'SSB0382', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x30f161d90>, 'text': 'dan4 zhe4 yi4 dian2 yi3 jing1 zu2 gou4 biao3 ming2 juan4 fu2 de5 chao1 gao1 ren2 qi4', 'age_group': 'B', 'gender': 'female', 'accent': 'north'}
{'id': 'SSB03410187', 'speaker_id': 'SSB0341', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x30f1ec550>, 'text': 'zhong1 lian2 hang2 jiang1 bei4 xiao1 jian3 bai3 fen1 zhi1 shi2 de5 fei1 xing2 xiao3 shi2 shu4', 'age_group': 'C', 'gender': 'female', 'accent': 

In [61]:
for split in ["train", "validation", "test"]:
    hanzi_ids = hanzi_dataset[split]["id"]
    pinyin_ids = pinyin_dataset[split]["id"]

    assert hanzi_ids == pinyin_ids

print("Hanzi and Pinyin datasets use identical splits.")

Hanzi and Pinyin datasets use identical splits.


In [62]:
train_ids = set(hanzi_dataset["train"]["id"])
validation_ids = set(hanzi_dataset["validation"]["id"])
test_ids = set(hanzi_dataset["test"]["id"])

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

print("No utterance overlap between train, validation, and test.")

No utterance overlap between train, validation, and test.


In [ ]:
# from pathlib import Path

# PROCESSED_DATA_DIR = PROJECT_ROOT / "processed_data"
# PROCESSED_DATA_DIR.mkdir(
#     parents=True,
#     exist_ok=True,
# )

# hanzi_dataset_path = (
#     PROCESSED_DATA_DIR / "aishell3_hanzi"
# )

# pinyin_dataset_path = (
#     PROCESSED_DATA_DIR / "aishell3_pinyin"
# )

# hanzi_dataset.save_to_disk(
#     hanzi_dataset_path
# )

# pinyin_dataset.save_to_disk(
#     pinyin_dataset_path
# )

# print("Saved Hanzi dataset:", hanzi_dataset_path.resolve())
# print("Saved Pinyin dataset:", pinyin_dataset_path.resolve())

Saving the dataset (0/36 shards):   0%|          | 0/56935 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/6327 [00:00<?, ? examples/s]

Saving the dataset (0/15 shards):   0%|          | 0/24773 [00:00<?, ? examples/s]

Saving the dataset (0/36 shards):   0%|          | 0/56935 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/6327 [00:00<?, ? examples/s]

Saving the dataset (0/15 shards):   0%|          | 0/24773 [00:00<?, ? examples/s]

Saved Hanzi dataset: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/processed_data/aishell3_hanzi
Saved Pinyin dataset: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/processed_data/aishell3_pinyin


# Start to build CTC vocab. Must have blank, unk, and | to separete repeated phones. 
1. Hanzi: each token contains every character
2. pinyin: each token contains pinyin+tone
3. phone: in alphabet a to z

In [63]:
hanzi_characters = sorted({
    character
    for text in hanzi_dataset["train"]["text"]
    for character in text
}) # getting the unique characters in the training set

hanzi_vocab = {
    character: index
    for index, character in enumerate(hanzi_characters)
} # vocab to id

# adding special tokens for unknown and padding
hanzi_vocab["[UNK]"] = len(hanzi_vocab)
hanzi_vocab["[PAD]"] = len(hanzi_vocab)
hanzi_vocab["|"] = len(hanzi_vocab)

print("Hanzi vocabulary size:", len(hanzi_vocab))
print(list(hanzi_vocab.items())[:20])

Hanzi vocabulary size: 4015
[('一', 0), ('丁', 1), ('七', 2), ('万', 3), ('丈', 4), ('三', 5), ('上', 6), ('下', 7), ('不', 8), ('与', 9), ('丐', 10), ('丑', 11), ('专', 12), ('且', 13), ('丕', 14), ('世', 15), ('丘', 16), ('丙', 17), ('业', 18), ('丛', 19)]


In [64]:
def read_train_pinyin_tokens(phone_set_path):
    pinyin_tokens = []
    reading_train_section = False

    with Path(phone_set_path).open(
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            line = line.strip()

            if line == "# The phones frequencies of the train set.":
                reading_train_section = True
                continue

            if line == "# The phones frequencies of the test set.":
                break

            if not reading_train_section:
                continue

            if not line or line.startswith("#"):
                continue

            pinyin_token, frequency = line.split()

            if not re.fullmatch(r"[a-z]+[1-5]", pinyin_token):
                raise ValueError(
                    f"Invalid Pinyin token: {pinyin_token}"
                )

            pinyin_tokens.append(pinyin_token)

    if len(pinyin_tokens) != len(set(pinyin_tokens)):
        raise ValueError(
            "Duplicate Pinyin tokens found in training section."
        )

    return sorted(pinyin_tokens)

In [65]:
pinyin_tokens = read_train_pinyin_tokens(
    DATA_ROOT / "phone_set.txt"
)

pinyin_vocab = {
    token: index
    for index, token in enumerate(pinyin_tokens)
}

pinyin_vocab["|"] = len(pinyin_vocab)
pinyin_vocab["[UNK]"] = len(pinyin_vocab)
pinyin_vocab["[PAD]"] = len(pinyin_vocab)

print("Pinyin vocabulary size:", len(pinyin_vocab))
print(list(pinyin_vocab.items())[:20])

Pinyin vocabulary size: 1847
[('a1', 0), ('a2', 1), ('a3', 2), ('a4', 3), ('a5', 4), ('ai1', 5), ('ai2', 6), ('ai3', 7), ('ai4', 8), ('ai5', 9), ('an1', 10), ('an2', 11), ('an3', 12), ('an4', 13), ('ang1', 14), ('ang2', 15), ('ang4', 16), ('ao1', 17), ('ao2', 18), ('ao3', 19)]


In [66]:
phone_tokens = list("abcdefghijklmnopqrstuvwxyz12345")

phone_vocab = {
    token: index
    for index, token in enumerate(phone_tokens)
}

phone_vocab["|"] = len(phone_vocab)
phone_vocab["[UNK]"] = len(phone_vocab)
phone_vocab["[PAD]"] = len(phone_vocab)

print("Phone vocabulary size:", len(phone_vocab))
print(phone_vocab)

Phone vocabulary size: 34
{'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, '1': 26, '2': 27, '3': 28, '4': 29, '5': 30, '|': 31, '[UNK]': 32, '[PAD]': 33}


Question: How does CTC handle Chinese characters that have multiple pronunciations (polyphonic characters)?
CTC + n-gram: how to sampling

In [67]:
import json
from pathlib import Path

vocab_names = ["hanzi", "pinyin", "phone"]

vocab_list = [hanzi_vocab, pinyin_vocab, phone_vocab]

VOCAB_DIR = Path("vocab")
VOCAB_DIR.mkdir(parents=True, exist_ok=True)

for vocab_name, vocabulary in zip(vocab_names, vocab_list):
    vocab_path = VOCAB_DIR / f"{vocab_name}_vocab.json"

    with vocab_path.open("w", encoding="utf-8") as vocab_file:
        json.dump(
            vocabulary,
            vocab_file,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"Saved {vocab_name} vocabulary: "
        f"{vocab_path} "
        f"({len(vocabulary)} tokens)"
    )

Saved hanzi vocabulary: vocab/hanzi_vocab.json (4015 tokens)
Saved pinyin vocabulary: vocab/pinyin_vocab.json (1847 tokens)
Saved phone vocabulary: vocab/phone_vocab.json (34 tokens)


# CTC tokenizer

In [72]:
from transformers import Wav2Vec2CTCTokenizer

hanzi_vocab_path = VOCAB_DIR / "hanzi_vocab.json"
assert hanzi_vocab_path.exists(), (
    f"Vocabulary not found: {hanzi_vocab_path.resolve()}"
)

phone_vocab_path = VOCAB_DIR / "phone_vocab.json"
assert phone_vocab_path.exists(), (
    f"Vocabulary not found: {phone_vocab_path.resolve()}"
)

hanzi_tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(hanzi_vocab_path),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

phone_tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(phone_vocab_path),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

print("Tokenizer size:", len(hanzi_tokenizer))
print("Vocab file:", hanzi_vocab_path.resolve())
print("Phone Tokenizer size:", len(phone_tokenizer))
print("Phone Vocab file:", phone_vocab_path.resolve())

Tokenizer size: 4017
Vocab file: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/vocab/hanzi_vocab.json
Phone Tokenizer size: 36
Phone Vocab file: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/vocab/phone_vocab.json


In [74]:
class PinyinSyllableCTCTokenizer(Wav2Vec2CTCTokenizer):
    def _tokenize(self, text, **kwargs):
        syllables = text.lower().split()
        tokens = []

        for index, syllable in enumerate(syllables):
            if index > 0:
                tokens.append(self.word_delimiter_token)

            tokens.append(syllable)

        return tokens

pinyin_tokenizer = PinyinSyllableCTCTokenizer(
    vocab_file=str(VOCAB_DIR / "pinyin_vocab.json"),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

In [71]:
hanzi_example = hanzi_dataset["train"][0]["text"]

hanzi_encoding = hanzi_tokenizer(hanzi_example)

print("Text:", hanzi_example)
print("IDs:", hanzi_encoding.input_ids[:20])
print(
    "Tokens:",
    hanzi_tokenizer.convert_ids_to_tokens(
        hanzi_encoding.input_ids[:20]
    ),
)

Text: 但这一点已经足够表明卷福的超高人气
IDs: [151, 3512, 0, 2161, 1053, 2739, 3418, 775, 3199, 1600, 440, 2560, 2410, 3413, 3936, 93, 1879]
Tokens: ['但', '这', '一', '点', '已', '经', '足', '够', '表', '明', '卷', '福', '的', '超', '高', '人', '气']


In [92]:
# check the decoding of the tokenized output
pinyin_example_1 = "er2 zi5 wanr2 shou3 ji1"
pinyin_example_2 = "er2zi5wanr2shou3ji1"

encoding_1 = pinyin_tokenizer(pinyin_example_1)
encoding_2 = pinyin_tokenizer(pinyin_example_2)

tokens_1 = pinyin_tokenizer.convert_ids_to_tokens(
    encoding_1.input_ids
)
tokens_2 = pinyin_tokenizer.convert_ids_to_tokens(
    encoding_2.input_ids
)

decoded_1 = pinyin_tokenizer.decode(
    encoding_1.input_ids,
    group_tokens=False,
)

decoded_2 = pinyin_tokenizer.decode(
    encoding_2.input_ids,
    group_tokens=False,
)

print("Input:", pinyin_example_1)
print("Tokens:", tokens_1)
print("Decoded:", decoded_1)

print("Input:", pinyin_example_2)
print("Tokens:", tokens_2)
print("Decoded:", decoded_2)

Input: er2 zi5 wanr2 shou3 ji1
Tokens: ['er2', 'zi5', 'wanr2', 'shou3', 'ji1']
Decoded: er2zi5wanr2shou3ji1
Input: er2zi5wanr2shou3ji1
Tokens: ['er2', 'zi5', 'wanr2', 'shou3', 'ji1']
Decoded: er2zi5wanr2shou3ji1


# Processor, combined audio and text together

In [93]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16_000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

from transformers import Wav2Vec2Processor

hanzi_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=hanzi_tokenizer,
)

pinyin_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=pinyin_tokenizer,
)

# phone_processor = Wav2Vec2Processor(
#     feature_extractor=feature_extractor,
#     tokenizer=phone_tokenizer,
# )

In [94]:
def prepare_dataset(example, processor):
    audio = example["audio"]

    processed = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        text=example["text"],
    )

    example["input_values"] = (
        processed["input_values"][0]
    )

    example["input_length"] = len(
        example["input_values"]
    )

    example["labels"] = processed["labels"]

    return example

#{
#     "id": "SSB00050001",
#     "audio": {...},
#     "text": "广州女大学生……",
#     "input_values": [...],
#     "input_length": 123456,
#     "labels": [...]
# }

In [95]:
prepared_hanzi_dataset = hanzi_dataset.map(
    prepare_dataset,
    fn_kwargs={
        "processor": hanzi_processor,
    }, desc="Preparing Hanzi dataset",)

prepared_pinyin_dataset = pinyin_dataset.map(
    prepare_dataset,
    fn_kwargs={
        "processor": pinyin_processor,
    },
    desc="Preparing Pinyin dataset"
)

# prepared_phone_dataset = phone_dataset.map(prepare_dataset, fn_kwargs = {"processor": phone_processor}, desc = "Preparing Phone Dataset")

Preparing Hanzi dataset:   0%|          | 0/56935 [00:00<?, ? examples/s]

Preparing Hanzi dataset:   0%|          | 0/6327 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [88]:
test_example = prepare_dataset(
    hanzi_dataset["train"][0],
    hanzi_processor,
)

print("Input length:", test_example["input_length"])
print("Number of labels:", len(test_example["labels"]))
print("First input values:",
      test_example["input_values"][:10])
print("Labels:", test_example["labels"])

NameError: name 'prepare_dataset' is not defined

# Check function to see if the train set and test set share the same speaker


In [23]:
# check if the train and test set share the same speakers, if so, raise an error
train_speakers = set(aishell_base["train"]["speaker_id"])
test_speakers = set(aishell_base["test"]["speaker_id"])

if train_speakers & test_speakers:
    raise ValueError("Train and test sets share the same speakers!")

ValueError: Train and test sets share the same speakers!

# data collator
